# Random Forest Evaluation

This notebook evaluates the saved Random Forest model separately from the production XGBoost pipeline. It uses the existing patient-level splits, a probability threshold of 0.60, and the two-consecutive-observation alert rule. No production files are changed.

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'model_ready'
MODEL_PATH = ROOT / 'models' / 'random_forest.pkl'
THRESHOLD = 0.60
CONSECUTIVE = 2
print('Project:', ROOT)
print('Model exists:', MODEL_PATH.exists())

Project: e:\Earlysepsispredicton
Model exists: True


In [2]:
model = joblib.load(MODEL_PATH)
frames = {name: pd.read_csv(DATA / f'{name}.csv') for name in ['train', 'validation', 'test']}
feature_names = list(getattr(model, 'feature_names_in_', []))
if not feature_names:
    feature_names = [c for c in frames['train'].columns if c not in {'Patient_ID', 'Dataset', 'SepsisLabel'}]
missing = [c for c in feature_names if c not in frames['train'].columns]
if missing:
    raise ValueError(f'Model features missing from data: {missing[:5]}')
print('Features:', len(feature_names))
for name, frame in frames.items():
    print(f'{name}: {len(frame):,} rows, {frame.Patient_ID.nunique():,} patients, positives={int(frame.SepsisLabel.sum()):,}')

Features: 196
train: 1,086,436 rows, 28,235 patients, positives=19,560
validation: 231,472 rows, 6,050 patients, positives=4,162
test: 234,302 rows, 6,051 patients, positives=4,194


In [3]:
def evaluate_patient_level(frame, probability, threshold=THRESHOLD, consecutive=CONSECUTIVE):
    work = frame[['Patient_ID', 'ICULOS', 'SepsisLabel']].copy()
    work['probability'] = np.asarray(probability)
    work = work.sort_values(['Patient_ID', 'ICULOS'])
    work['hit'] = (work['probability'] >= threshold).astype(int)
    work['consecutive_alert'] = work.groupby('Patient_ID')['hit'].transform(lambda s: s.rolling(consecutive, min_periods=consecutive).sum().ge(consecutive).astype(int))
    truth = work.groupby('Patient_ID').SepsisLabel.max().astype(int)
    pred = work.groupby('Patient_ID').consecutive_alert.max().reindex(truth.index).fillna(0).astype(int)
    patient_probability = work.groupby('Patient_ID').probability.max().reindex(truth.index)
    tn, fp, fn, tp = confusion_matrix(truth, pred, labels=[0, 1]).ravel()
    return {
        'patients': int(len(truth)), 'accuracy': accuracy_score(truth, pred),
        'precision': precision_score(truth, pred, zero_division=0),
        'recall': recall_score(truth, pred, zero_division=0),
        'f1': f1_score(truth, pred, zero_division=0),
        'specificity': tn / (tn + fp) if tn + fp else 0.0,
        'roc_auc': roc_auc_score(truth, patient_probability),
        'pr_auc': average_precision_score(truth, patient_probability),
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
    }

results = {}
for name, frame in frames.items():
    x = frame[feature_names].apply(pd.to_numeric, errors='coerce').fillna(0).astype('float32')
    probability = model.predict_proba(x)[:, 1]
    results[name] = evaluate_patient_level(frame, probability)
    del x
pd.DataFrame(results).T.round(4)

,patients,accuracy,precision,recall,f1,specificity,roc_auc,pr_auc,tp,tn,fp,fn
train,28235.0,0.9956,0.9735,0.9664,0.9699,0.9979,0.9996,0.9969,1983.0,26129.0,54.0,69.0
validation,6050.0,0.9372,0.8061,0.1795,0.2937,0.9966,0.7579,0.3550,79.0,5591.0,19.0,361.0
test,6051.0,0.9392,0.8103,0.2136,0.3381,0.9961,0.7765,0.4061,94.0,5589.0,22.0,346.0
